<a href="https://colab.research.google.com/github/author-sanjay/AirSafetyAI/blob/Data-Normalization/AirCraftAccidentDataAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
from bs4 import BeautifulSoup
import time
import csv
from google.colab import drive
from collections import defaultdict
import json
import pandas as pd
import re
import numpy as np
from datetime import datetime

# Data Collection

###### Data collection has been done already from airsafety db from year 2000 to 2025 resulting up to 6500+ recorded incidents found to train the model. Please note that this data is only being used for research purposes and model training

# Data Processing

✈️ Data Preprocessing & Normalization Notes
1. Data Cleaning



*   Find & delete duplicates → Remove duplicate entries for consistency.
*   Fix unknown/missing dates → Handle invalid or missing dates.
*   Normalize time values → Standardize time formats.
*   Merge Date & Time → Create a single DateTime column.
* Convert to UTC → Ensure uniform time reference across all records.
* Normalize flight hours → Standardize numeric flight time fields.



2. Dropping Irrelevant / Biased Fields

* Drop operator (Owner/Operator) → Not useful for prediction.

* Drop MSN & Registration → Identifiers, no predictive value.

* Drop raw DateTime after merging/UTC conversion → Avoid redundancy.

* Drop investigating agency info → Administrative, not predictive.

* Drop airports (Departure, Destination) → Route-level details handled separately in model logic.

* Drop Confidence rating & Location → Not meaningful for model training.

* Drop fatalities/occupants counts → Would bias model (seriousness ≠ death toll).

3. Normalizing Categories

* Aircraft Damage → Collapse variations into None, Minor, Substantial, Destroyed, Missing, Unknown.

* Phase → Normalize to Takeoff, Initial climb, En route, Approach, Landing, Unknown.

* Remove ground-only phases (Taxi, Pushback/Towing, Standing).

* Nature of Flight → Collapse into broader groups:

* Passenger (all types)

* Cargo/Ferry

* Military/Govt

* Training/Test/Calibration

* Special Ops (firefighting, agricultural, parachuting, ambulance, survey, patrol)

* Other/Unknown/Illegal

4. Narrative (Keep for NLP)

* Retain Narrative text → Used for NLP to extract accident cause/trigger (birdstrike, hydraulic failure, structural issue, etc.).

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
file_path = "/content/drive/MyDrive/accidents.json"

# Load your data
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Dictionary to track occurrences
seen = defaultdict(list)

for idx, entry in enumerate(data):
    # Composite key: Date + Time + Registration + Location
    key = f"{entry.get('Date','')}_{entry.get('Time','')}_{entry.get('Registration','')}_{entry.get('Location','')}"
    seen[key].append(idx)

# Find duplicates (keys with more than 1 entry)
duplicates = {k: v for k, v in seen.items() if len(v) > 1}


In [4]:


file_path = "/content/drive/MyDrive/accidents.json"
cleaned_file_path = "/content/drive/MyDrive/accidents_cleaned.json"
csv_file_path = "/content/drive/MyDrive/accidents_cleaned.csv"

#Load JSON into pandas
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)



# Handle unk. date YYYY
def normalize_date(val):
    if pd.isna(val):
        return None
    val = str(val).strip()
    match = re.match(r"unk\. date (\d{4})", val, flags=re.IGNORECASE)
    if match:
        year = int(match.group(1))
        return f"{year}-12-31"

    try:
        return pd.to_datetime(val, errors="coerce").strftime("%Y-%m-%d")
    except Exception:
        return None

df["Date"] = df["Date"].apply(normalize_date)

def normalize_time(val):
    val = str(val).strip()
    if not val:
        return "12:00"
    val = val.replace("LT", "").strip()

    try:
        t = pd.to_datetime(val, format="%H:%M", errors="coerce")
        if pd.isna(t):
            return "12:00"
        return t.strftime("%H:%M")
    except:
        return "12:00"

df["Time"] = df["Time"].apply(normalize_time)


df["Datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    errors="coerce"
)

df.drop(columns=["Date", "Time"], inplace=True)

df["Datetime"] = df["Datetime"].dt.tz_localize("UTC")

def normalize_category(cat):
    if pd.isna(cat):
        return "Incident"
    cat = str(cat).strip().lower()
    if cat == "accident":
        return "Accident"
    elif cat == "incident":
        return "Incident"
    elif cat == "serious incident":
        return "Serious Incident"
    elif cat == "unlawful interference":
        return "Unlawful Interference"
    elif cat == "uk":
        return "Unknown"
    elif cat == "other":
        return "Other"
    else:
        return "Incident"

df["Category"] = df["Category"].apply(normalize_category)

df.drop(columns=["Owner/operator"], inplace=True)

df.drop(columns=["Registration", "MSN"], inplace=True)

def clean_hours(x):
    if pd.isna(x) or str(x).strip() == "":
        return np.nan
    try:
        return float(str(x).split()[0].replace(",", ""))
    except:
        return np.nan

df["Total airframe hrs"] = df["Total airframe hrs"].apply(clean_hours)

def bucketize_hours(x):
    if pd.isna(x):
        return "Mid-life"
    elif x < 5000:
        return "New"
    elif x < 20000:
        return "Mid-life"
    else:
        return "Old"

df["airframe_bucket"] = df["Total airframe hrs"].apply(bucketize_hours)

df["Year of manufacture"] = pd.to_numeric(df["Year of manufacture"], errors="coerce")
df["aircraft_age"] = df["Datetime"].dt.year - df["Year of manufacture"]
df.loc[df["Year of manufacture"].isna(), "aircraft_age"] = np.nan
df.drop(columns=["Datetime"], inplace=True)
df.drop(columns=["Investigating agency"], inplace=True)
df.drop(columns=["Destination airport","Departure airport"], inplace=True)
df.drop(columns=["Confidence Rating","Location"], inplace=True)
df.drop(columns=["Fatalities","Other fatalities"], inplace=True)


phase_mapping = {
    "Approach": "Approach",
    "Landing": "Landing",
    "En route": "Cruise",
    "Taxi": "Ground",
    "Take off": "Takeoff",
    "Initial climb": "Takeoff",
    "Pushback / towing": "Ground",
    "Standing": "Ground",
    "Manoeuvring  (airshow, firefighting, ag.ops.)": "Special ops",
    "Unknown": "Unknown",
    "": "Unknown"
}

df["Phase"] = df["Phase"].map(lambda x: phase_mapping.get(str(x).strip(), "Unknown"))
df = df[df["Phase"] != "Ground"].reset_index(drop=True)


nature_mapping = {
    "Passenger - Scheduled": "Passenger",
    "Passenger - Non-Scheduled/charter/Air Taxi": "Passenger",
    "Passenger": "Passenger",
    "Executive": "Passenger",
    "Cargo": "Cargo",
    "Military": "Military",
    "Private": "Private",
    "Training": "Training",
    "Ferry/positioning": "Ferry/Positioning",
    "Fire fighting": "Special Operations",
    "Parachuting": "Special Operations",
    "Agricultural": "Special Operations",
    "Ambulance": "Special Operations",
    "Survey": "Special Operations",
    "Aerial patrol": "Special Operations",
    "Test": "Test/Demo",
    "Calibration/Inspection": "Test/Demo",
    "Demo/Airshow/Display": "Test/Demo",
    "Illegal Flight": "Illegal",
    "Unknown": "Unknown",
    "": "Unknown",
    "-": "Unknown",
    "SF": "Unknown"
}

df["Nature"] = df["Nature"].replace(nature_mapping)


# Extracting Incident Category Using NLP

### Source of Classification Data (ADREP / ECCAIRS)

The classification data we are using comes from internationally recognized aviation safety taxonomies:

* **ICAO / SKYbrary website**
Provides the official ADREP taxonomy (Accident/Incident Data Reporting) maintained by ICAO. This taxonomy defines standard categories and codes for classifying the causes, contributing factors, and types of aviation accidents and incidents.

*  **ECCAIRS (European Coordination Centre for Aviation Incident Reporting Systems)**
ECCAIRS publishes Data Definition Standards (DDS) that implement the ADREP taxonomy in structured form (attributes like Events, Occurrence Categories, Occurrence Classes, etc.). These standards are widely used by national investigation authorities and safety databases around the world.

In our dataset, we extracted ~96 distinct events from these standards. Each event includes:

1. ATA Code (system/component reference)

2. Event Name

3. Description

4. Broader Group (e.g., Flight Controls, Engine/Powerplant, Fire/Explosion)

5. Approach Category (our added field to help downstream airport-approach decision models)

### Why This Matters

We could have simply used an accidents.json file (with only past cases) and classified narratives against those. But that would give us pre-structured data limited to accidents that have already happened.

By instead using the official ICAO/ECCAIRS classification standards:

* We cover all known types of aviation events (not just those present in our dataset).

* We can train or fine-tune models to recognize potential causes of incidents, even those that have not yet occurred in the historical record.

* We align with international safety reporting practices, so our outputs are compatible with ICAO, IATA, and national investigation bodies.

### Why We Need This Taxonomy

* It provides a hierarchical, standardized vocabulary for aviation incidents.

* Enables NLP models to classify narratives into specific causes (e.g., “rudder malfunction”) and broader categories (e.g., “System/Component Failure – Non-Powerplant”).

* Supports predictive analysis: training models not just on what happened before, but on what can happen based on aviation safety knowledge.

In [5]:

# Load taxonomy events
with open("/content/adrep_events.json", "r", encoding="utf-8") as f:
    taxonomy = json.load(f)

candidate_causes = [e["name"] for e in taxonomy]


Compute Embeddings for Events

In [6]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)  # GPU if available


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [7]:
def classify_batch(narratives, candidate_causes):
    results = []
    for text in narratives:
        if not isinstance(text, str) or text.strip() == "":
            results.append({"narrative": text, "matched_event": "Unknown", "score": 0.0})
            continue

        res = classifier(text, candidate_causes, multi_label=False)
        top_label = res["labels"][0]
        top_score = float(res["scores"][0])

        # Map back to taxonomy entry
        event = next((e for e in taxonomy if e["name"] == top_label), None)
        results.append({
            "narrative": text,
            "matched_event": top_label,
            "ata_code": event["ata_code"] if event else None,
            "broad_group": event["broad_group"] if event else None,
            "approach": event["approach"] if event else None,
            "score": top_score
        })
    return results



In [ ]:
# Take first 10 narratives
sample_narratives = df["Narrative"].iloc[:10].tolist()

sample_results = classify_batch(sample_narratives, candidate_causes)

# Pretty print results
import pandas as pd
sample_df = pd.DataFrame(sample_results)
print(sample_df[["narrative", "matched_event", "ata_code", "broad_group", "approach", "score"]])


                                           narrative  \
0  The pilot-in-command (PIC) stated he was in cr...   
1                             Damaged beyond repair.   
2  While performing the ILS runway 18 approach to...   
3  The Bandeirante aircraft was coming in to land...   
4  Disappeared near the border of the Angolan pro...   
5  NWL100, a Beech King Air with 2 crew, was trai...   
6  On approach to Penang runway 22, the aircraft ...   
7  At 17:00 Saab 340 HB-AKK arrived at Zurich, Sw...   
8  One flight attendant sustained serious injurie...   
9  The Shorts 360 plane had been leased to Sirte ...   

                                 matched_event ata_code           broad_group  \
0        Aircraft wing structure related event     5700    Structural/Systems   
1                         Degraded performance     0101  Aerodrome/Navigation   
2              Aircraft lighting related event     3300                 Other   
3    Aircraft empennage stucture related event     5500    

In [ ]:
import json
import pandas as pd
from transformers import pipeline

# -------- Load data ----------
file_path = "/content/drive/MyDrive/accidents.json"

with open(file_path, "r", encoding="utf-8") as f:
    accidents = json.load(f)

df = pd.DataFrame(accidents)

# -------- Load taxonomy ----------
with open("/content/adrep_events.json", "r", encoding="utf-8") as f:
    taxonomy = json.load(f)

candidate_causes = [e["name"] for e in taxonomy]

# -------- Classifier pipeline ----------
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli",
                      device=0)  # GPU if available

def classify_batch(narratives, candidate_causes):
    results = []
    for text in narratives:
        if not isinstance(text, str) or text.strip() == "":
            results.append({
                "matched_event": "Unknown",
                "ata_code": None,
                "broad_group": None,
                "approach": None,
                "score": 0.0
            })
            continue

        res = classifier(text, candidate_causes, multi_label=False)
        top_label = res["labels"][0]
        top_score = float(res["scores"][0])
        event = next((e for e in taxonomy if e["name"] == top_label), None)

        results.append({
            "matched_event": top_label,
            "ata_code": event["ata_code"] if event else None,
            "broad_group": event["broad_group"] if event else None,
            "approach": event["approach"] if event else None,
            "score": top_score
        })
    return results


# -------- Process in batches ----------
batch_size = 10

for i in range(0, len(df), batch_size):
    batch = df.iloc[i:i+batch_size]

    # Find which rows still need processing
    to_process_idx = []
    for idx, row in batch.iterrows():
        if "matched_event" not in row or pd.isna(row["matched_event"]):
            to_process_idx.append(idx)

    if not to_process_idx:
        print(f"Batch {i}-{i+batch_size} already done, skipping.")
        continue

    narratives = df.loc[to_process_idx, "Narrative"].tolist()
    results = classify_batch(narratives, candidate_causes)

    # Assign results back to dataframe
    for j, idx in enumerate(to_process_idx):
        for key, value in results[j].items():
            df.at[idx, key] = value

    # Save progress back to JSON every batch
    df.to_json(file_path, orient="records", indent=2, force_ascii=False)
    print(f"Processed batch {i}-{i+batch_size}, saved to {file_path}")


Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Processed batch 0-10, saved to /content/drive/MyDrive/accidents.json
Processed batch 10-20, saved to /content/drive/MyDrive/accidents.json
Processed batch 20-30, saved to /content/drive/MyDrive/accidents.json
Processed batch 30-40, saved to /content/drive/MyDrive/accidents.json
Processed batch 40-50, saved to /content/drive/MyDrive/accidents.json
Processed batch 50-60, saved to /content/drive/MyDrive/accidents.json
Processed batch 60-70, saved to /content/drive/MyDrive/accidents.json
Processed batch 70-80, saved to /content/drive/MyDrive/accidents.json
Processed batch 80-90, saved to /content/drive/MyDrive/accidents.json
Processed batch 90-100, saved to /content/drive/MyDrive/accidents.json
Processed batch 100-110, saved to /content/drive/MyDrive/accidents.json
Processed batch 110-120, saved to /content/drive/MyDrive/accidents.json
Processed batch 120-130, saved to /content/drive/MyDrive/accidents.json
Processed batch 130-140, saved to /content/drive/MyDrive/accidents.json
Processed ba